# **EXPLAINABLE ASPECT BASED SENTIMENT ANALYSIS ON HINDI TEXT USING SHAP AND LIME**

# **Import Libraries**

In [1]:
import pandas as pd
import torch
import pickle
import numpy as np
import random
import re
import unicodedata
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

# **Load Dataset**

In [2]:
import gdown

file_id = "1LqvWpNOulpPYNbfkxUcAVBK0W5R6K7BZ"
url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(url, "combined_hindi_absa.csv", quiet=False)

df = pd.read_csv("combined_hindi_absa.csv")

print("Dataset shape:", df.shape)

Downloading...
From: https://drive.google.com/uc?id=1LqvWpNOulpPYNbfkxUcAVBK0W5R6K7BZ
To: /content/combined_hindi_absa.csv
100%|██████████| 1.40M/1.40M [00:00<00:00, 64.6MB/s]

Dataset shape: (5470, 4)


# **Data Preprocessing**

In [3]:
def clean_text(text):

    text = str(text)

    text = unicodedata.normalize("NFC", text)

    text = re.sub(r"http\S+|www\S+", "", text)

    text = re.sub(r"<.*?>", "", text)

    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    text = re.sub(r"[^\w\s\u0900-\u097F.,!?]", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text


df["sentence"] = df["sentence"].apply(clean_text)
df["aspect"] = df["aspect"].apply(clean_text)

# **Create Input Text with Aspect Highlighting**

In [4]:
def highlight_aspect(row):

    sentence = row["sentence"]
    aspect = row["aspect"]

    highlighted_sentence = sentence.replace(
        aspect,
        f"[ASP] {aspect} [ASP]"
    )

    input_text = f"aspect: {aspect} sentence: {highlighted_sentence}"

    return input_text


df["input_text"] = df.apply(highlight_aspect, axis=1)

print(df["input_text"].head())

0    aspect: फ्रंट कैमरा sentence: [ASP] फ्रंट कैमर...
1    aspect: डिजाइन sentence: मुझे [ASP] डिजाइन [AS...
2    aspect: फ्रंट कैमरा sentence: [ASP] फ्रंट कैमर...
3    aspect: परफॉरमेंस sentence: [ASP] परफॉरमेंस [A...
4    aspect: चार्जिंग sentence: [ASP] चार्जिंग [ASP...
Name: input_text, dtype: object


# **Encode Labels**

In [5]:
label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(df["polarity"])

print("\nLabel Mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(i, "->", label)


Label Mapping:
0 -> negative
1 -> neutral
2 -> positive


# **80 / 10 / 10 Dataset Split**

In [6]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("\nDataset Split Sizes")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


Dataset Split Sizes
Train: 4376
Validation: 547
Test: 547


In [7]:
train_dataset = Dataset.from_pandas(
    train_df[["input_text","label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["input_text","label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["input_text","label"]],
    preserve_index=False
)

# **Load MuRIL Model**

In [8]:
model_name = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

In [10]:
!pip install torchinfo

In [9]:
import torch
from torchinfo import summary

summary(
    model,
    input_data={
        "input_ids": torch.randint(0, model.config.vocab_size, (1, 128), dtype=torch.long),
        "attention_mask": torch.ones((1, 128), dtype=torch.long),
    },
    depth=3,
    col_names=("input_size", "output_size", "num_params", "trainable"),
)

Layer (type:depth-idx)                                  Input Shape               Output Shape              Param #                   Trainable
BertForSequenceClassification                           --                        [1, 3]                    --                        True
├─BertModel: 1-1                                        [1, 128]                  [1, 768]                  --                        True
│    └─BertEmbeddings: 2-1                              --                        [1, 128, 768]             --                        True
│    │    └─Embedding: 3-1                              [1, 128]                  [1, 128, 768]             151,514,880               True
│    │    └─Embedding: 3-2                              [1, 128]                  [1, 128, 768]             1,536                     True
│    │    └─Embedding: 3-3                              [1, 128]                  [1, 128, 768]             393,216                   True
│    │    └─LayerNorm:

# **Class Weight**

In [10]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
weights_tensor_cpu = torch.tensor(class_weights, dtype=torch.float32)

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        weights_tensor_on_device = weights_tensor_cpu.to(model.device)

        # Apply the weighted CrossEntropyLoss
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor_on_device)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# **Tokenization**

In [11]:
def tokenize_function(example):

    return tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","label"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids","attention_mask","label"]
)

Map:   0%|          | 0/4376 [00:00<?, ? examples/s]

Map:   0%|          | 0/547 [00:00<?, ? examples/s]

Map:   0%|          | 0/547 [00:00<?, ? examples/s]

# **Evaluation Metrics**

In [12]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = logits.argmax(axis=1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# **Training Arguments**

In [13]:
training_args = TrainingArguments(

    output_dir="./muril_results",

    learning_rate=1e-5,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    num_train_epochs=10,

    eval_strategy="epoch",

    logging_strategy="epoch",

    warmup_steps=137,

    lr_scheduler_type="linear",

    fp16=True
)

# **Trainer**

In [14]:
# Use this instead of the standard Trainer
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# **Train Model**

In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.089226,1.023948,0.769653,0.770292,0.769653,0.766709
2,0.942037,0.854988,0.835466,0.835821,0.835466,0.834290
3,0.793918,0.736203,0.846435,0.845902,0.846435,0.845166
4,0.672166,0.652366,0.855576,0.854786,0.855576,0.854710
5,0.586511,0.602010,0.850091,0.849972,0.850091,0.848873
6,0.520811,0.567326,0.846435,0.846643,0.846435,0.844831
7,0.465747,0.544454,0.848263,0.848015,0.848263,0.847426
8,0.426830,0.525886,0.853748,0.853438,0.853748,0.853137
9,0.399942,0.516510,0.853748,0.853426,0.853748,0.852868
10,0.384730,0.515644,0.850091,0.850308,0.850091,0.849154


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1370, training_loss=0.6281918769335225, metrics={'train_runtime': 398.1755, 'train_samples_per_second': 109.901, 'train_steps_per_second': 3.441, 'total_flos': 2878460789944320.0, 'train_loss': 0.6281918769335225, 'epoch': 10.0})

# **Testing the model**

In [18]:
from transformers import AutoModelForSequenceClassification

checkpoint_path = "./muril_results/checkpoint-1000"

model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)


print("\nFinal Test Evaluation")

test_metrics = trainer.evaluate(test_dataset)

print(test_metrics)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Final Test Evaluation


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
No log,0.573716,0,0.820841,0.821445,0.820841,0.821103


{'eval_loss': 0.573716402053833, 'eval_accuracy': 0.8208409506398537, 'eval_precision': 0.8214449170573486, 'eval_recall': 0.8208409506398537, 'eval_f1': 0.8211028315077088}


# **Sample Predictions**

In [ ]:
predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=1)

print("\nSample Predictions:\n")

for i in range(10):

    sentence = test_df.iloc[i]["sentence"]
    aspect = test_df.iloc[i]["aspect"]

    true_label = label_encoder.inverse_transform([test_df.iloc[i]["label"]])[0]
    pred_label = label_encoder.inverse_transform([preds[i]])[0]

    print("Sentence:", sentence)
    print("Aspect:", aspect)
    print("True:", true_label)
    print("Pred:", pred_label)
    print()


Sample Predictions:

Sentence: निर्देशक और लेखक ने पटकथा पर अधिक मेहनत नहीं की है।
Aspect: निर्देशक
True: negative
Pred: negative

Sentence: जेड1 कॉम्पैक्ट की तुलना में, इसमें थोड़ा सुधार हुआ है खास तौर से शॉट लेने के टाइम के मामले में, या कैमरा एप्प खुलने के मामले में।
Aspect: कैमरा एप्प
True: positive
Pred: positive

Sentence: निकॉन कूल पिक्स S6300 Nikon Coolpix S6300 कीमत 9,652 रुपए, कैमरा साइज 93.6 x 57.7 x 36 एमएम, वेट 160 ग्राम रेज्यूलूशन 16 मेगापिक्सल फोकल लेंथ 25250 एमएम ऑप्टिकल जूम 10 एक्स एलसीडी डिस्प्ले 2.7 इंच 230 डॉट्स आईएसओ 1253200 वीडियो रिकार्डिंग 1920 x 1080 रेज्यूलूशन कूल पिक्स एस 6300 ट्रैवलिंग लवर्स के लिए अच्छा केमरा है।
Aspect: फोकल लेंथ
True: positive
Pred: positive

Sentence: इसमें स्क्रीन पर एक साथ दो एप्स चला सकते हैं।
Aspect: स्क्रीन
True: neutral
Pred: neutral

Sentence: ये अल्ट्रा HD यूजर्स को 8 मिलियन पिक्सल और फुल HD की 2 मिलियन पिक्सल क्वालिटी देती है।
Aspect: पिक्सल क्वालिटी
True: neutral
Pred: neutral

Sentence: ऐसा इसलिए भी है क्योंकि इस घड़ी का बॉडी

# **Saving The Model and Tokenizer**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r /content/muril_results/checkpoint-1370 /content/drive/MyDrive/MuRIL_ABSA_10

In [ ]:
tokenizer.save_pretrained("/content/drive/MyDrive/MuRIL_ABSA_10")

('/content/drive/MyDrive/MuRIL_ABSA_10/tokenizer_config.json',
 '/content/drive/MyDrive/MuRIL_ABSA_10/tokenizer.json')